In [8]:
%%time
import h5py

# Extracted matrix - First gene 
print("OLD MATRIX")
print("**********")

with h5py.File('192k_biosamples_57k_genes_correctedMatrix_LW20230125.h5', 'r') as f:
    Datasetnames=f.keys()
    print(Datasetnames)
    matrix = f['matrix'][0]
    corrected_matrix = f['corrected_matrix'][0]
    
    #sample_names = f['sample_names'][:]
    gene_length = f['gene_length'][:]
    gene_names = f['gene_names'][:]
    
print(matrix.shape)
print()

print(gene_names[0])
print(gene_length[0])    
print(matrix)
print(corrected_matrix)
print()

OLD MATRIX
**********
<KeysViewHDF5 ['corrected_matrix', 'gene_length', 'gene_names', 'matrix', 'sample_names']>
(192601,)

b'A1BG'
3374.0
[  0   0   0 ... 482 320 308]
[0.         0.         0.         ... 0.14285715 0.09484292 0.09128631]

CPU times: user 26.2 ms, sys: 6.75 ms, total: 32.9 ms
Wall time: 230 ms


In [11]:
%%time
import h5py

# Extracted matrix - First gene 
print("5M EXTRACTED MATRIX")
print("*******************")

with h5py.File('192k_biosamples_57k_genes_correctedMatrix_LW20230125.h5', 'r') as f:
    Datasetnames=f.keys()
    print(Datasetnames)
    matrix = f['qn_matrix'][:3]
    
print(matrix.shape)
print(matrix)
print()


5M EXTRACTED MATRIX
*******************
<KeysViewHDF5 ['corrected_matrix', 'gene_length', 'gene_names', 'matrix', 'qn_matrix', 'sample_names', 'threshold_matrix']>
(3, 87875)
[[0.00030363 0.00495186 0.00581927 ... 0.15882927 0.08592538 0.10288213]
 [0.00267582 0.00029755 0.00206837 ... 0.08055572 0.07328279 0.09075348]
 [0.00246942 0.00269969 0.00175528 ... 0.00712895 0.00256089 0.00221092]]

CPU times: user 2.81 ms, sys: 2.28 ms, total: 5.09 ms
Wall time: 13.4 ms


In [9]:
%%time 
from itertools import islice
from collections import Counter
import numpy as np
import h5py
import re
import json

# Open ARCHS4 v2.1 Data
infile = "/home/ajoshi/archs4/data/archs4_gene_human_v2.1.h5"

# - Use read_direct method to improve read performance
with h5py.File(infile, 'r') as hdf5_file:
    data = hdf5_file['data']
    expression = data['expression'][21433]
    symbols = hdf5_file['meta']['genes']['gene_symbol'][21433]

print(symbols)
print(expression[0])
print(expression[-1])

b'ENSG00000271983'
10
10
CPU times: user 22 s, sys: 1.99 s, total: 24 s
Wall time: 42.4 s


In [10]:
matrix

array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.4285715e-01,
        9.4842918e-02, 9.1286309e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 6.3511834e-02,
        7.8870900e-02, 7.7210464e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 5.1042573e-03,
        2.7150305e-03, 1.6290182e-03],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.1290000e+03,
        2.1030000e+03, 1.5220000e+03],
       [0.0000000e+00, 0.0000000e+00, 1.0000000e+00, ..., 2.2100000e+03,
        3.2760000e+03, 2.2350000e+03]], dtype=float32)

In [ ]:
len(matrix)

In [8]:
%%time
with h5py.File('192k_biosamples_57k_genes_extractedMatrix_LW20230122.h5', 'r') as f:
    sample_names = f['sample_names'][:]
    gene_names = f['gene_names'][:]

print(gene_names[0])

b'ENSG00000271983'
CPU times: user 98.9 ms, sys: 3.68 s, total: 3.78 s
Wall time: 4.29 s


In [9]:
gene_names

array([b'ENSG00000271983', b'LINC01132', b'ENSG00000229642', ...,
       b'ENSG00000288049', b'PHF21B', b'NMT1'], dtype=object)

In [10]:
b'A1BG'in gene_names

True

In [12]:
gene_names.index(b'A1BG')

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [15]:
import numpy as np
np.where(gene_names == b'A1BG')[0]

array([3500])

In [22]:
#!/usr/bin/env python
# coding: utf-8

# In[ ]:

import csv
import json
import pickle
import h5py
import numpy as np

# Open ARCHS4 v2.1 Data
infile = "/home/ajoshi/archs4/data/archs4_gene_human_v2.1.h5"

# Open the JSON file - Selected Samples
root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/sbatch_files/results/"
with open(root + 'allSelected_uniqueSampleIdx_list_LW20230121.json') as json_file:
    # Load the JSON data from the file
    sampleIdxs_list = json.load(json_file)

# Open the csv file - Selected Genes 
gene_names = []
gene_idxs = []

with open("/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/data_results/edited_gene_name_length_idx_data.csv", 'r') as file:
    reader = csv.reader(file)
    # Skip the header row
    next(reader)
    for row in reader:
        gene_names.append(str(row[1]))
        gene_idxs.append(int(row[4]))

# Get the selected samples - names 
pickle_in = open("/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/Analysis/afterReadFilteration_CategorySampleSizes/archs4_sampleNames_list.pkl", 'rb')
sample_names_list = pickle.load(pickle_in)
selected_sample_names = [sample_names_list[i] for i in sampleIdxs_list]

# Read ARCHS4 Matrix
with h5py.File(infile, 'r') as h5py_file:
    data = h5py_file['data']
    matrix = data['expression'][:]

# Extract the rows and columns specified by the indexes
extracted_matrix = matrix[gene_idxs, :][:, sampleIdxs_list]

# Create a new HDF5 file
with h5py.File('extracted_matrix.h5', 'w') as f:
    
    # Create a dataset for the matrix
    dset = f.create_dataset('matrix', data=extracted_matrix)
     # Create a dataset for the sample names
    dset2 = f.create_dataset('sample_names', (len(selected_sample_names),), dtype=h5py.special_dtype(vlen=str))
    dset2[:] = selected_sample_names
    # Create a dataset for the gene names
    dset3 = f.create_dataset('gene_names', (len(gene_names),), dtype=h5py.special_dtype(vlen=str))
    dset3[:] = gene_names



In [23]:
gene_names

['PCAT1',
 'METTL9',
 'ENSG00000285935',
 'ENSG00000278607',
 'SMIM11',
 'RNA5SP315',
 'DEPDC1-AS1',
 'ENSG00000187904',
 'ENSG00000238745',
 'EARS2',
 'FLOT2',
 'ENSG00000256325',
 'MRPL51P2',
 'COL27A1',
 'ENSG00000274321',
 'DPP10-AS1',
 'ENSG00000228793',
 'IL1R1-AS1',
 'ENSG00000271691',
 'ENSG00000232050',
 'TM9SF4',
 'ENSG00000224560',
 'ENSG00000258460',
 'USP20',
 'ENSG00000286764',
 'FAM47DP',
 'SRGAP3-AS4',
 'IGHD5-12',
 'ELOA2',
 'MBOAT2',
 'ENSG00000254578',
 'MCF2',
 'LINC02753',
 'RNU4-21P',
 'PPP1R14A',
 'TSHB',
 'TXLNB',
 'GJA4',
 'NIPBL-DT',
 'TM4SF20',
 'LINC00330',
 'ENSG00000232692',
 'JPT1',
 'PHB1P20',
 'SEPTIN7P4',
 'ENSG00000229721',
 'FHAD1',
 'ENSG00000271420',
 'TTYH3',
 'ENSG00000235005',
 'SERTAD3-AS1',
 'ENSG00000261938',
 'RBM38-AS1',
 'ENSG00000286840',
 'MIR577',
 'TRAV5',
 'FOSL2',
 'LINC02006',
 'ENSG00000256894',
 'ENSG00000287384',
 'ARHGAP27',
 'PSMA1P1',
 'RNU6-1120P',
 'KRTAP9-10P',
 'MSTO1',
 'ENSG00000254787',
 'ALPP',
 'ENSG00000253997',
 'TY

In [20]:
symbols

array([b'A1BG', b'A1BG-AS1'], dtype=object)

In [18]:
%%time
import h5py
import json
import numpy as np

# Extracted matrix - First gene 
print("OLD MATRIX")
print("**********")

with h5py.File('192k_biosamples_57k_genes_correctedMatrix_LW20230125.h5', 'r') as f:
    matrix = f['matrix'][:]
    
    readCount_list = np.sum(matrix, axis = 0)
    sample_names = f['sample_names'][:]
    
print(len(readCount_list))
sample_names_decoded = [x.decode('UTF-8') for x in sample_names]
readCount_list = readCount_list.tolist()
data = dict(zip(sample_names_decoded, readCount_list))

with open("archs4_192kSamples_matchedGenes_readCountSum.json", "w+") as f:
    json.dump(data, f)

OLD MATRIX
**********
192601
CPU times: user 14.4 s, sys: 33.3 s, total: 47.7 s
Wall time: 11min 4s


In [17]:

readCount_list[0:10]


array([  32906,   48865,   61634,   55349,  490356,  183527,   42104,
         10870, 1337780,   16359], dtype=uint64)